# S07: S05a residual signal as continuous weights with a no-trade band

Pre-registration: [`S07_README.md`](S07_README.md). Engine: `src/equity_trading/s07.py`.

Every tradable stock holds a contrarian weight ∝ −clip(s, ±3), scaled to 100% stock gross, hedged in its sector ETF on actual holdings. Stocks trade only when the target leaves a no-trade band, and only to the band edge. Market orders at the next open, S05b measured costs. Two pre-registered arms: band = 0.5× and 1.0× the average position size.

In [ ]:
N_PLACEBO = 500              # per arm; gates use alpha = 0.025 (two arms)
N_JOBS = -1
PER_YEAR, WINDOW_S = 4, 3.0  # S05b quote samples (reused from the cache)

import json, os, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import __version__, s04, s05, s05b, s07
from equity_trading.alpaca import ALPACA_VERSION, download_years, sample_quote_spreads
from equity_trading.data import vendor_dir, work_dir

SYNTHETIC = os.environ.get("S07_SYNTHETIC") == "1"
YEARS = list(range(2020, 2026))                      # 2026 not used
CFG = s07.S07Config(n_placebo=N_PLACEBO)
ARMS = {"band05": 0.5, "band10": 1.0}
if SYNTHETIC:
    YEARS, N_PLACEBO = [2024, 2025], 10
    CFG = replace(CFG, w_cap=0.05)                   # 40 synthetic stocks: average weight 2.5%

tag = ("synthetic_" if SYNTHETIC else "") + f"alpaca{ALPACA_VERSION}_s07{s07.S07_VERSION}_p{s04.PANEL_VERSION}"
OUT = ROOT / "results" / ("s07_synthetic" if SYNTHETIC else "s07")
OUT.mkdir(parents=True, exist_ok=True)
CACHE = work_dir() / "results" / "s07" / tag
CACHE.mkdir(parents=True, exist_ok=True)
QUOTE_CACHE = vendor_dir("Alpaca") / "quote_samples" / f"{CFG.timeframe}_w{WINDOW_S:g}s"
QUOTE_CACHE.mkdir(parents=True, exist_ok=True)

COL = {"band05": "#2a78d6", "band10": "#1baf7a", "band00": "#eb6834", "grey": "#8a8a84"}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e6e3", "grid.linewidth": 0.6, "axes.titleweight": "bold", "font.size": 9})

if SYNTHETIC:
    from equity_trading.universe import SECTOR_ETF_01, UNIVERSE_01
    ETF_OF = {s: SECTOR_ETF_01[k] for k, v in UNIVERSE_01.items() for s in v}
    BENCH = "XLK"
else:
    constituents = pd.read_csv(ROOT / "reference" / "sp500_constituents_20260913.csv")
    ETF_OF = dict(zip(constituents["Symbol"], constituents["GICS Sector"].map(s05.SECTOR_ETF)))
    BENCH = "SPY"
STOCKS = sorted(ETF_OF)
ETFS = sorted(set(ETF_OF.values()))
print(f"synthetic={SYNTHETIC} | stocks={len(STOCKS)} | arms={ARMS} | placebos per arm={N_PLACEBO}")
print(CFG)

## 1. Data and measured costs

In [ ]:
t0 = time.time()
if SYNTHETIC:
    panel, _ = s04.synthetic_ohlc_panel(YEARS)
else:
    download_years(STOCKS + ETFS + [BENCH], CFG.timeframe, YEARS, workers=8)
    panel = s04.cached_panel(STOCKS + ETFS + [BENCH], CFG.timeframe, YEARS, ffill_bars=CFG.ffill_bars)
have = set(panel.symbols)
TRADED = [s for s in STOCKS if s in have and ETF_OF[s] in have]
SC = np.array([panel.symbols.index(s) for s in TRADED])
EC = np.array([panel.symbols.index(ETF_OF[s]) for s in TRADED])
QUOTED = sorted(set(TRADED) | {ETF_OF[s] for s in TRADED})
print(f"panel {panel.c.shape} in {time.time() - t0:.0f}s | traded stocks {len(TRADED)}")
SESSIONS = s05b.sample_sessions(YEARS, PER_YEAR, seed=42)
if SYNTHETIC:
    samples = pd.DataFrame([(s, 3e-4, 10, k, d) for d in SESSIONS for k in range(26) for s in QUOTED],
                           columns=["symbol", "half_spread", "n_quotes", "slot", "date"])
else:
    missing = [d for d in SESSIONS if not (QUOTE_CACHE / f"{d}.parquet").exists()]
    print(f"quote sessions cached: {len(SESSIONS) - len(missing)}/{len(SESSIONS)}")
    for d in missing:
        sample_quote_spreads(QUOTED, d, s05b.slot_starts(YEARS, d), window_s=WINDOW_S, cache_dir=QUOTE_CACHE)
    samples = pd.concat([pd.read_parquet(QUOTE_CACHE / f"{d}.parquet") for d in SESSIONS], ignore_index=True)
H = s05b.cost_array(panel.index, panel.day_start, panel.symbols, s05b.cost_model(samples))
print(f"measured half-spread: stocks {np.median(H[:, SC]) * 1e4:.2f} bps, ETFs {np.median(H[:, np.unique(EC)]) * 1e4:.2f} bps")

est_path = CACHE / "params.npz"
if est_path.exists():
    P = dict(np.load(est_path))
else:
    t0 = time.time(); P = s05.estimate_all(panel, SC, EC, CFG.s05(), s07.first_decision_day(CFG))
    np.savez(est_path, **P); print(f"estimated in {time.time() - t0:.0f}s")

## 2. Backtests

Each arm at 1×, 2× and 0× measured cost, plus the no-band reference (band 0, reported only).

In [ ]:
runs = {}
t0 = time.time()
for arm, bm in {**ARMS, "band00": 0.0}.items():
    for cm in (1.0, 2.0, 0.0):
        if arm == "band00" and cm == 2.0:
            continue
        runs[(arm, cm)] = s07.backtest(panel, P, SC, EC, H, replace(CFG, band_mult=bm, cost_mult=cm), record=(cm == 1.0))
START = runs[("band05", 1.0)]["first_day"]
print(f"{len(runs)} backtests in {(time.time() - t0) / 60:.1f} min; trading starts {START.date()}")
rows = {}
for (arm, cm), r in runs.items():
    m = s04.perf_metrics(r["daily_equity"], CFG.initial_equity, START)
    ts = s07.turnover_stats(r, START, CFG.initial_equity)
    rows[f"{arm} {cm:g}x"] = {**m, **ts}
for arm in [*ARMS, "band00"]:
    g = runs[(arm, 1.0)]["gross_exposure"]
    eq = runs[(arm, 1.0)]["bar_equity"].to_numpy()
    live = g > 0
    rows[f"{arm} 1x"]["avg_gross_exposure"] = float(np.mean(g[live] / eq[live]))
metrics = pd.DataFrame(rows).T
display(metrics.round(4))
print("S05a reference (thresholds, same signal): gross Sharpe 1.22 at 0x; S05b net -25.3% with measured costs")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for arm in [*ARMS, "band00"]:
    for cm, ls in ((1.0, "-"), (0.0, ":")):
        e = runs[(arm, cm)]["daily_equity"]; e = e[e.index >= START]
        ax[0].plot(e.index, e, color=COL[arm], ls=ls, lw=0.9, label=f"{arm} {cm:g}x")
ax[0].set_title("Equity (solid net, dotted gross)"); ax[0].legend(frameon=False, fontsize=7)
for arm in ARMS:
    e = runs[(arm, 1.0)]["daily_equity"]; e = e[e.index >= START]
    ax[1].fill_between(e.index, e / e.cummax() - 1, 0, color=COL[arm], alpha=0.5, lw=0, label=arm)
ax[1].set_title("Drawdown (net)"); ax[1].legend(frameon=False)
plt.tight_layout(); plt.show()
by_year = {}
for arm in ARMS:
    e = runs[(arm, 1.0)]["daily_equity"]; e = e[e.index >= START]
    ye = e.groupby(e.index.year).last()
    by_year[arm] = ye / ye.shift(1).fillna(CFG.initial_equity) - 1
display(pd.DataFrame(by_year).T.round(4))
for arm in ARMS:
    runs[(arm, 1.0)]["daily_equity"].to_csv(OUT / f"daily_equity_{arm}.csv", header=["equity"], float_format="%.2f")

## 3. Placebo: shuffled signal (each day, stocks receive another tradable stock's s-score path)

In [ ]:
from joblib import Parallel, delayed
placebos, ranks = {}, {}
for arm, bm in ARMS.items():
    cfg_a = replace(CFG, band_mult=bm)
    pl_path = CACHE / f"placebo_shuffle_{arm}_{N_PLACEBO}.parquet"
    if pl_path.exists():
        pl = pd.read_parquet(pl_path)
    else:
        t0 = time.time()
        pl = pd.DataFrame(Parallel(n_jobs=N_JOBS, batch_size=2, verbose=5)(
            delayed(s07.run_placebo)(panel, P, SC, EC, H, cfg_a, CFG.seed * 1000 + i, START) for i in range(N_PLACEBO)))
        pl.to_parquet(pl_path); print(f"{arm}: {N_PLACEBO} placebos in {(time.time() - t0) / 60:.1f} min")
    pl.to_csv(OUT / f"placebo_shuffle_{arm}.csv", index=False, float_format="%.6g")
    placebos[arm] = pl
    bm_ = metrics.loc[f"{arm} 1x"]
    ranks[arm] = {k: s04.placebo_rank(bm_[k], pl[k].to_numpy()) for k in ("total_return", "sharpe")}
display(pd.concat({a: pd.DataFrame(v).T for a, v in ranks.items()}).round(4))
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
for i, arm in enumerate(ARMS):
    ax[i].hist(placebos[arm]["sharpe"], bins=40, color=COL["grey"], alpha=0.7, label="shuffled")
    ax[i].axvline(metrics.loc[f"{arm} 1x", "sharpe"], color=COL[arm], lw=2, label="strategy")
    ax[i].set_title(f"{arm}: Sharpe vs shuffled signal"); ax[i].legend(frameon=False)
plt.tight_layout(); plt.show()

## 4. Bootstrap, market exposure, gates

In [ ]:
days = pd.to_datetime(panel.days)
bench_close = pd.Series(panel.c[panel.day_start[1:] - 1, panel.symbols.index(BENCH)], index=days)
G, boots, mkts = {}, {}, {}
for arm in ARMS:
    e = runs[(arm, 1.0)]["daily_equity"]; e = e[e.index >= START]
    rd = e.pct_change().dropna().to_numpy()
    boots[arm] = {b: s04.block_bootstrap_mean(rd, b, 10_000, CFG.seed) for b in (10, 20, 40)}
    mkts[arm] = s04.market_regression(e, bench_close)
    G[arm] = s07.gates(boots[arm], ranks[arm], metrics.loc[f"{arm} 2x"].to_dict(), metrics.loc[f"{arm} 1x"].to_dict())
display(pd.concat({a: pd.DataFrame(b, index=["mean", "lo95", "hi95"]).T for a, b in boots.items()}))
print("market regression vs", BENCH, {a: {k: round(v, 5) for k, v in m.items()} for a, m in mkts.items()})
display(pd.DataFrame(G))
summary = {"study": "S07", "synthetic": SYNTHETIC, "years": YEARS, "config": CFG.as_dict(), "arms": ARMS,
           "trading_start": str(START.date()), "n_placebo_per_arm": N_PLACEBO, "metrics": metrics.to_dict(orient="index"),
           "placebo_shuffle_ranks": ranks, "bootstrap": {a: {str(k): v for k, v in b.items()} for a, b in boots.items()},
           "market": mkts, "gates": G,
           "versions": {"package": __version__, "alpaca": ALPACA_VERSION, "s05": s05.S05_VERSION, "s07": s07.S07_VERSION,
                        "panel": s04.PANEL_VERSION}}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=float))
(OUT / "manifest.json").write_text(json.dumps({"files": sorted(p.name for p in OUT.iterdir()), "cache": str(CACHE)}, indent=2))
print(json.dumps({a: {k: g[k] for k in ["statistical_verdict", "economic_verdict", "verdict"]} for a, g in G.items()}, indent=2))